# Phase 1D.3-Lite Colab smoke
Run all cells on a Tesla T4 after providing the approved repository URL and immutable ref. This demo does not complete official Phase 1D.3.

In [ ]:
# Lite setup: create a unique, non-destructive run and checkout an immutable commit.
import os, subprocess, sys, json, re, time, getpass, platform, pathlib, uuid
LITE_TASK_FILTER = 'ADMIN_FEEDBACK_GROUPING_STAGE1'  # use 'all' to run all six tasks
RUN_LIVE_NESTJS_SMOKE = False  # prior live smoke already passed
ENABLE_LITE_DIAGNOSTICS = True
run_id = f"{time.strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:8]}"
lite_evidence_dir = pathlib.Path('/content/lite-evidence') / run_id
repo = pathlib.Path('/content') / f'dacn-lite-{run_id}'
def block(message):
    lite_evidence_dir.mkdir(parents=True, exist_ok=True)
    (lite_evidence_dir/'blocked.json').write_text(json.dumps({'status':'BLOCKED','diagnostic':str(message),'runId':run_id}, indent=2)+'\n')
    raise RuntimeError(str(message))
REPO_URL = os.getenv('DACN_REPO_URL') or input('Owner-provided repository URL: ').strip()
REPO_REF = os.getenv('DACN_REPO_REF') or input('Owner-provided immutable 40-character commit SHA: ').strip().lower()
if not REPO_URL or not re.fullmatch(r'[0-9a-f]{40}', REPO_REF):
    block('BLOCKED: provide a repository URL and immutable 40-character commit SHA')
try:
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', REPO_REF], check=True)
    source_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    if source_commit != REPO_REF:
        block('BLOCKED: checked-out source SHA does not match the requested immutable SHA')
except Exception as exc:
    block(f'BLOCKED: source checkout failed: {type(exc).__name__}')

In [ ]:
# Verify the T4 and preserve Colab's existing Torch installation before any package install.
import subprocess, json, sys, pathlib, platform
if platform.machine() not in ("x86_64", "amd64"):
    block("BLOCKED: Lite Node install supports Colab Linux x64 only")
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,uuid,memory.total", "--format=csv,noheader"], text=True).strip().splitlines()
    torch_before = subprocess.check_output([sys.executable, "-c", "import torch; print(torch.__version__); print(torch.__file__); print(torch.cuda.is_available())"], text=True).strip().splitlines()
except Exception as exc:
    block(f"BLOCKED: GPU/Torch prerequisites unavailable: {type(exc).__name__}")
if len(gpu) != 1 or "T4" not in gpu[0]:
    block(f"BLOCKED: expected one Tesla T4, found {gpu}")
gpu_name, gpu_uuid, gpu_memory = [item.strip() for item in gpu[0].split(",", 2)]
if len(torch_before) != 3 or torch_before[2] != "True":
    block("BLOCKED: existing Colab Torch/CUDA unavailable")
constraint = pathlib.Path("/content/lite-torch-constraint.txt")
constraint.write_text(f"torch=={torch_before[0]}\n")
report = pathlib.Path("/content/lite-pip-report.json")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--report", str(report), "-r", str(repo/"ai/requirements-lite.txt"), "-c", str(constraint)], check=True)
    subprocess.run([sys.executable, '-c', f"import sys; sys.path.insert(0, {str(repo)!r}); from ai.benchmark.lite_smoke import validate_pip_report; validate_pip_report('/content/lite-pip-report.json')"], check=True)
except Exception as exc:
    block(f"BLOCKED: dry-run report missing, malformed, or unsafe: {type(exc).__name__}")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo/"ai/requirements-lite.txt"), "-c", str(constraint)], check=True)
torch_after = subprocess.check_output([sys.executable, "-c", "import torch; print(torch.__version__); print(torch.__file__); print(torch.cuda.is_available())"], text=True).strip().splitlines()
if torch_after != torch_before:
    block("BLOCKED: existing Torch changed")


In [ ]:
# Install supported Node/pnpm and only the API workspace dependencies.
import subprocess, shutil, re, pathlib, os, platform
node = shutil.which('node')
node_ok = False
if node:
    version = subprocess.check_output([node, '--version'], text=True).strip().lstrip('v').split('.')
    node_ok = int(version[0]) == 24 and (int(version[1]), int(version[2])) >= (18, 0)
if not node_ok:
    archive = '/content/node-v24.18.0-linux-x64.tar.xz'
    subprocess.run(['curl', '-fsSL', 'https://nodejs.org/dist/v24.18.0/node-v24.18.0-linux-x64.tar.xz', '-o', archive], check=True)
    subprocess.run(['tar', '-xJf', archive, '-C', '/content'], check=True)
    os.environ['PATH'] = '/content/node-v24.18.0-linux-x64/bin:' + os.environ['PATH']
corepack_dir = pathlib.Path('/content/lite-corepack-bin')
corepack_dir.mkdir(exist_ok=True)
subprocess.run(['corepack', 'enable', '--install-directory', str(corepack_dir)], cwd=repo, check=True)
os.environ['PATH'] = str(corepack_dir) + os.pathsep + os.environ['PATH']
subprocess.run(['corepack', 'prepare', 'pnpm@10.12.1', '--activate'], cwd=repo, check=True)
subprocess.run(['pnpm', 'install', '--frozen-lockfile', '--filter', '@dacn/api...'], cwd=repo, check=True)
subprocess.run(['pnpm', '--filter', '@dacn/api', 'exec', 'prisma', 'generate'], cwd=repo, check=True)

In [ ]:
# Start the actual FastAPI worker once; keep the token hidden and wait for lifespan READY.
import subprocess, os, pathlib, time, re, getpass, json, atexit
lite_evidence_dir.mkdir(parents=True, exist_ok=True)
if 'stop_runtime' in globals():
    stop_runtime()
server = None
log = None
def stop_runtime():
    global server, log
    if server is not None and server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=15)
    if log is not None and not log.closed:
        log.flush()
        log.close()
atexit.register(stop_runtime)
runtime_token = getpass.getpass('AI runtime token (hidden): ')
if not runtime_token:
    block('BLOCKED: runtime token is required')
env = os.environ.copy()
env.update({'AI_RUNTIME_TOKEN': runtime_token, 'AI_RUNTIME_GPU_UUID': gpu_uuid,
            'AI_RUNTIME_MODEL_ID': 'Qwen/Qwen2.5-1.5B-Instruct',
            'AI_RUNTIME_MODEL_REVISION': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
            'AI_RUNTIME_LOAD_MODE': 'fp16'})
if ENABLE_LITE_DIAGNOSTICS:
    env.update({'AI_RUNTIME_LITE_DIAGNOSTICS': '1',
                'AI_RUNTIME_LITE_DIAGNOSTICS_DIR': str(lite_evidence_dir/'diagnostics')})
log = open(lite_evidence_dir/'uvicorn.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'ai.runtime.app:app', '--host', '127.0.0.1', '--port', '8000'], cwd=repo, env=env, stdout=log, stderr=subprocess.STDOUT)
deadline = time.time() + 600
while time.time() < deadline:
    log.flush()
    contents = (lite_evidence_dir/'uvicorn.log').read_text(errors='replace')
    if 'Application startup complete' in contents:
        break
    if server.poll() is not None:
        stop_runtime()
        block(f'BLOCKED: Uvicorn startup failed; inspect {lite_evidence_dir}/uvicorn.log')
    time.sleep(2)
else:
    stop_runtime()
    block(f'BLOCKED: worker did not reach lifespan READY; inspect {lite_evidence_dir}/uvicorn.log')
match = re.findall(r'AI model loaded in ([0-9.]+) seconds', contents)
if not match:
    stop_runtime()
    block(f'BLOCKED: model-load duration missing; inspect {lite_evidence_dir}/uvicorn.log')
model_load_seconds = float(match[-1])

In [ ]:
# Re-run only Stage 1 for diagnosis; opt into all tasks/NestJS with the environment flags below.
import subprocess, pathlib, json, os, importlib.metadata as metadata
outdir = lite_evidence_dir/'attempts'
outdir.mkdir(parents=True, exist_ok=True)
evidence = {'runId': run_id, 'sourceCommit': source_commit,
            'modelId': 'Qwen/Qwen2.5-1.5B-Instruct', 'modelRevision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
            'gpu': gpu_name, 'gpuUuid': gpu_uuid, 'gpuMemory': gpu_memory,
            'torchVersion': torch_after[0], 'torchPath': torch_after[1], 'modelLoadSeconds': model_load_seconds,
            'dependencyVersions': {name: metadata.version(name) for name in ('fastapi', 'uvicorn', 'jsonschema', 'transformers', 'accelerate')}}
env.update({'AI_RUNTIME_BASE_URL': 'http://127.0.0.1:8000', 'AI_RUNTIME_TOKEN': runtime_token,
            'AI_RUNTIME_MODEL_ID': evidence['modelId'], 'AI_RUNTIME_MODEL_REVISION': evidence['modelRevision'],
            'AI_RUNTIME_GPU_NAME': gpu_name, 'AI_RUNTIME_GPU_MEMORY': gpu_memory,
            'AI_RUNTIME_TORCH_VERSION': torch_after[0], 'AI_RUNTIME_TORCH_PATH': torch_after[1],
            'AI_RUNTIME_MODEL_LOAD_SECONDS': str(model_load_seconds)})
task_filter = LITE_TASK_FILTER
fixtures = [json.loads(line) for line in (repo/'ai/benchmark/fixtures/lite-smoke.jsonl').read_text().splitlines() if line.strip()]
selected_fixtures = fixtures if task_filter == 'all' else [row for row in fixtures if row['task'] == task_filter]
smoke = {'status': 'BLOCKED', 'diagnostic': 'smoke runner did not complete'}
live = {'status': 'NOT_RUN', 'diagnostic': 'NestJS live smoke not requested for this run'}
try:
    if not selected_fixtures:
        raise ValueError(f'no Lite fixture matches task filter {task_filter!r}')
    subprocess.run(['pnpm', '--filter', '@dacn/api', 'exec', 'tsx', '../../ai/benchmark/validate-inputs.ts'], cwd=repo, input=json.dumps(selected_fixtures, ensure_ascii=False), text=True, check=True)
    command = [sys.executable, '-m', 'ai.benchmark.lite_smoke', '--output-dir', str(outdir)]
    if task_filter != 'all':
        command.extend(['--task', task_filter])
    run = subprocess.run(command, cwd=repo, env=env, text=True, capture_output=True)
    print(run.stdout)
    try:
        smoke = json.loads(run.stdout)
        if run.returncode and smoke.get('status') in ('PASS', 'PARTIAL_PASS'):
            smoke['status'] = 'FAIL'
            smoke['diagnostic'] = f'smoke runner exited {run.returncode}'
    except json.JSONDecodeError:
        smoke = {'status': 'FAIL', 'diagnostic': f'smoke runner exited {run.returncode}: {run.stderr[-4000:].replace(runtime_token, "[REDACTED]")}'}
    if RUN_LIVE_NESTJS_SMOKE:
        api_env = env.copy()
        api_env.update({'AI_RUNTIME_LIVE_SMOKE': '1', 'AI_RUNTIME_BASE_URL': 'http://127.0.0.1:8000',
                        'AI_RUNTIME_PROMPT_VERSIONS': subprocess.check_output(
                            [sys.executable, '-c', 'import json; from ai.runtime.prompts import PROMPT_VERSIONS; print(json.dumps(PROMPT_VERSIONS))'],
                            cwd=repo, text=True).strip()})
        live_run = subprocess.run(['pnpm', '--filter', '@dacn/api', 'exec', 'jest', '--runInBand', 'src/ai/ai-runtime-wire.spec.ts', '-t', 'live Colab runtime'],
                                  cwd=repo, env=api_env, text=True, capture_output=True, check=False)
        live = {'status': 'PASS' if live_run.returncode == 0 else 'FAIL', 'exitCode': live_run.returncode,
                'stdout': live_run.stdout[-8000:].replace(runtime_token, '[REDACTED]'),
                'stderr': live_run.stderr[-8000:].replace(runtime_token, '[REDACTED]')}
except Exception as exc:
    failure = f'{type(exc).__name__}: {str(exc).replace(runtime_token, "[REDACTED]")}'
    if smoke.get('status') == 'BLOCKED':
        smoke['diagnostic'] = failure
    else:
        live = {'status': 'FAIL', 'diagnostic': failure}
finally:
    evidence.update({'selectedTasks': smoke.get('selectedTasks', []), 'smokeStatus': smoke.get('status', 'FAIL'),
                     'smokeDiagnostic': smoke.get('diagnostic'), 'nestjsLive': live})
    (lite_evidence_dir/'colab-evidence.json').write_text(json.dumps(evidence, indent=2)+'\n')
    stop_runtime()
print('Evidence saved at', lite_evidence_dir)
if smoke.get('status') not in ('PASS', 'PARTIAL_PASS') or live.get('status') in ('FAIL', 'BLOCKED'):
    raise RuntimeError(f"Lite run incomplete: tasks={smoke.get('status')}, NestJS={live.get('status')}; see {lite_evidence_dir}")